# KL-IG Evaluation v3 — Tiered ImageNet Benchmark

Refactor of `evaluationv2.ipynb` onto **ImageNet validation subset** with **tiered evaluation** (compute budget scales with per-metric cost):

| Tier | n  | Metrics                                                                                           |
|------|----|---------------------------------------------------------------------------------------------------|
| **A** | 100 | Sensitivity-n, IROF, Occlusion, Insertion/Deletion (blur + noise), Complexity, Signed insertion, Sign agreement |
| **B** |  20 | Max-Sensitivity Robustness, Perturbation Robustness, Semantic Focus Score                        |
| **C** |  10 | KL-IG sign stability (MC reruns), Completeness vs MC samples                                     |

Tier C ⊂ Tier B ⊂ Tier A, so attributions only need to be computed **once** for the 100 Tier-A images. Results are **checkpointed to disk** — re-running does not recompute KL-IG.

Six methods compared: **KL-IG, IDG, ExpGrad, IG-zero, SmoothGrad, Vanilla Grad**.


## 1. Setup & imports


In [ ]:
!git clone https://github.com/Shameen5375/KLIG_V1.git 2>/dev/null || echo "Repo already cloned"
!pip install -q captum datasets opencv-python-headless


In [ ]:
import os, sys, math, json, pickle, warnings
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torchvision.models import ResNet50_Weights, resnet50
from scipy import stats
from tqdm.auto import tqdm

# Locate klig source tree (Colab clones to /content/KLIG_V1)
ROOT = Path.cwd()
for candidate in [ROOT, ROOT / "infocube-main",
                  Path("/content/KLIG_V1/infocube-main"),
                  Path("/content/KLIG_V1")]:
    if (candidate / "klig").exists():
        ROOT = candidate
        break
sys.path.append(str(ROOT))

from klig.image.attribution import ImageAttributor
from klig.image.stopping import find_sigma_stop
from klig.compare.captum_baselines import (
    run_ig, run_smoothgrad, run_expected_gradients, _absmax_collapse,
)
from klig.image.viz import _attr_to_rgb
from captum.attr import IntegratedGradients, Saliency

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Root:   {ROOT}")


## 2. Config & tier sizes

Scaled down by 10× from the full recommendation — change `TIER_A / TIER_B / TIER_C` to scale up.


In [ ]:
# ──────────────────────────────────────────────
# TIER SIZES (Tier C ⊂ Tier B ⊂ Tier A)
# ──────────────────────────────────────────────
TIER_A = 100   # cheap metrics
TIER_B = 20    # expensive metrics (need attribution reruns)
TIER_C = 10    # KL-IG variance / convergence
assert TIER_A >= TIER_B >= TIER_C

# ──────────────────────────────────────────────
# ImageNet source (first available wins)
# ──────────────────────────────────────────────
IMAGENET_ROOT   = os.environ.get("IMAGENET_ROOT", None)  # e.g. "/content/imagenet_val"
HF_DATASET_NAME = "imagenet-1k"                           # needs `huggingface-cli login`
HF_SPLIT        = "validation"
FALLBACK_LOCAL_DIR = Path("/content/KLIG_V1/images")      # last resort

# ──────────────────────────────────────────────
# Attribution hyperparameters
# ──────────────────────────────────────────────
N_STEPS        = 50         # KL-IG integration steps
N_SAMPLES      = 10         # KL-IG MC samples per step
SIGMA_FINAL    = 1 / 256
ADAPTIVE_SIGMA = True
IG_STEPS       = 50
BLUR_SIGMA     = 16.0
BLUR_KERNEL    = 51
SG_SAMPLES     = 50
EG_SAMPLES     = 50

# ──────────────────────────────────────────────
# Metric hyperparameters
# ──────────────────────────────────────────────
N_INSERTION_STEPS   = 100
N_SENS_SUBSETS      = 100
SENS_FRACTIONS      = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.8]
IROF_PATCH          = 14
IROF_STEPS          = 20
OCCLUSION_PATCH     = 14
OCCLUSION_STRIDE    = 7
ROB_EPS             = 0.02
ROB_TRIALS          = 5
PERTURBATION_SIGMAS = [0.01, 0.02, 0.05, 0.1, 0.2]
PERTURBATION_RUNS   = 3
CLIP_PCT            = 99.0

# ──────────────────────────────────────────────
# Methods + colors
# ──────────────────────────────────────────────
methods = ["KL-IG", "IDG", "ExpGrad", "IG-zero", "SmoothGrad", "Vanilla Grad"]
COLORS = {
    "KL-IG":         "#2E8B57",
    "IDG":           "#E07B39",
    "ExpGrad":       "#DC143C",
    "IG-zero":       "#7B68EE",
    "SmoothGrad":    "#1E90FF",
    "Vanilla Grad":  "#8B4513",
}

# ──────────────────────────────────────────────
# ImageNet normalization
# ──────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
TRANSFORM = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ──────────────────────────────────────────────
# Checkpointing
# ──────────────────────────────────────────────
CHECKPOINT_DIR = Path("eval_cache")
CHECKPOINT_DIR.mkdir(exist_ok=True)
ATTR_CACHE     = CHECKPOINT_DIR / "attributions.pkl"

print(f"Tiers: A={TIER_A}  B={TIER_B}  C={TIER_C}")
print(f"Cache: {CHECKPOINT_DIR.resolve()}")


## 3. Helpers: model, attribution methods, dispatch, boxplot util


In [ ]:
# ── Model ──
def load_model():
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights).to(DEVICE).eval()
    return model, weights.meta["categories"]


def denormalize(x):
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(-1, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=x.device).view(-1, 1, 1)
    if x.dim() == 4:
        mean, std = mean.unsqueeze(0), std.unsqueeze(0)
    return (x * std + mean).clamp(0, 1)


def predict_topk(model, x, k=5):
    with torch.no_grad():
        probs = model(x).softmax(-1)[0]
        top_p, top_i = probs.topk(k)
    return top_p.tolist(), top_i.tolist()


# ── Blur baseline ──
def make_blur_baseline(x, kernel_size=BLUR_KERNEL, sigma=BLUR_SIGMA):
    coords = torch.arange(kernel_size, dtype=torch.float32, device=x.device) - kernel_size // 2
    k1d = torch.exp(-0.5 * (coords / sigma) ** 2)
    k1d = k1d / k1d.sum()
    kh = k1d.view(1, 1, -1, 1).expand(3, -1, -1, -1)
    kw = k1d.view(1, 1, 1, -1).expand(3, -1, -1, -1)
    pad = kernel_size // 2
    out = F.conv2d(x, kh, padding=(pad, 0), groups=3)
    return F.conv2d(out, kw, padding=(0, pad), groups=3)


def make_eg_background(x, n=50):
    return torch.randn(n, *x.squeeze(0).shape, device=x.device)


# ──────────────────────────────────────────────
# 6 Attribution Methods (each returns (H, W) map)
# ──────────────────────────────────────────────
def compute_klig(model, x, target):
    sf = SIGMA_FINAL
    if ADAPTIVE_SIGMA:
        sf = max(find_sigma_stop(model, x, target=target, tau=0.95), 1.0 / 256.0)
    attr = ImageAttributor(model=model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                           sigma_final=sf, device=DEVICE)
    return attr.attribute(x, target=target, show_progress=False)


def compute_ig_zero(model, x, target):
    return run_ig(model, x, target, n_steps=IG_STEPS)


def compute_idg(model, x, target):
    x_inp = x.clone().requires_grad_(True)
    out = model(x_inp)
    out[0, target].backward()
    grad = x_inp.grad.detach()
    attr = x_inp.detach() * grad
    return _absmax_collapse(attr)


def compute_expgrad(model, x, target):
    bg = make_eg_background(x, n=EG_SAMPLES)
    return run_expected_gradients(model, x, target, background=bg, n_samples=EG_SAMPLES)


def compute_smoothgrad(model, x, target):
    return run_smoothgrad(model, x, target, n_samples=SG_SAMPLES)


def compute_vanilla_grad(model, x, target):
    sal = Saliency(model)
    attr = sal.attribute(x, target=target, abs=False)
    return _absmax_collapse(attr.detach())


COMPUTE_FN = {
    "KL-IG":         lambda m, x, t: compute_klig(m, x, t).attr_map("absmax"),
    "IDG":           compute_idg,
    "ExpGrad":       compute_expgrad,
    "IG-zero":       compute_ig_zero,
    "SmoothGrad":    compute_smoothgrad,
    "Vanilla Grad":  compute_vanilla_grad,
}


def compute_all(model, x, target):
    klig_res = compute_klig(model, x, target)
    maps = {"KL-IG": klig_res.attr_map("absmax")}
    for name in methods:
        if name == "KL-IG":
            continue
        maps[name] = COMPUTE_FN[name](model, x, target)
    return maps, klig_res


# ── Boxplot helper (reused across all metric cells) ──
def tier_boxplot(data, ylabel, title, methods=methods):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    bp = ax.boxplot([data[m] for m in methods], labels=methods, patch_artist=True,
                    medianprops=dict(color="black", lw=1.5))
    for patch, m in zip(bp["boxes"], methods):
        patch.set_facecolor(COLORS[m])
        patch.set_alpha(0.8)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=15)
    plt.tight_layout(); plt.show()


def tier_report(name, data, higher_better=True):
    """Print mean ± std per method, sorted by performance."""
    print(f"\n{name}  ({'higher = better' if higher_better else 'lower = better'})")
    rows = []
    for m in methods:
        arr = np.array(data[m])
        rows.append((m, arr.mean(), arr.std()))
    rows.sort(key=lambda r: -r[1] if higher_better else r[1])
    for m, mu, sd in rows:
        print(f"  {m:14s}: {mu:+.4f} ± {sd:.4f}")


print("Helpers ready — 6 methods:", methods)


## 4. Load stratified ImageNet validation subset

Tries three sources in order:

1. **`IMAGENET_ROOT` env var** → `torchvision.datasets.ImageFolder` layout (`<root>/<class_dir>/*.JPEG`)
2. **HuggingFace `datasets`** → `load_dataset("imagenet-1k", split="validation", streaming=True)` (requires `huggingface-cli login`)
3. **Local fallback** → any `*.jpg` / `*.png` under `/content/KLIG_V1/images`

Stratified sampling: **1 image per class** until `TIER_A` is filled.


In [ ]:
model, imagenet_labels = load_model()
print(f"Model: ResNet50 ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)")


def load_imagenet_subset(n_images):
    """Return list of (PIL.Image, class_id:int), stratified by class where possible."""
    # ── Source 1: torchvision ImageFolder ──
    if IMAGENET_ROOT and Path(IMAGENET_ROOT).exists():
        print(f"[dataset] ImageFolder @ {IMAGENET_ROOT}")
        from torchvision.datasets import ImageFolder
        ds = ImageFolder(IMAGENET_ROOT)
        by_class = defaultdict(list)
        for idx, (_, y) in enumerate(ds.samples):
            by_class[y].append(idx)
        rng = np.random.RandomState(0)
        class_order = sorted(by_class.keys())
        rng.shuffle(class_order)
        chosen = []
        for c in class_order:
            chosen.append(by_class[c][0])
            if len(chosen) >= n_images:
                break
        return [(ds[i][0].convert("RGB"), ds[i][1]) for i in chosen]

    # ── Source 2: HuggingFace datasets (streaming) ──
    try:
        from datasets import load_dataset
        print(f"[dataset] HuggingFace {HF_DATASET_NAME} [{HF_SPLIT}]")
        ds = load_dataset(HF_DATASET_NAME, split=HF_SPLIT, streaming=True)
        seen, out = set(), []
        for ex in ds:
            y = int(ex["label"])
            if y in seen:
                continue
            seen.add(y)
            out.append((ex["image"].convert("RGB"), y))
            if len(out) >= n_images:
                break
        if out:
            return out
    except Exception as e:
        print(f"[dataset] HF load failed: {e}")

    # ── Source 3: local fallback directory ──
    if FALLBACK_LOCAL_DIR.exists():
        print(f"[dataset] fallback: {FALLBACK_LOCAL_DIR}")
        paths = sorted(FALLBACK_LOCAL_DIR.glob("*.jpg")) + \
                sorted(FALLBACK_LOCAL_DIR.glob("*.png"))
        out = []
        for p in paths[:n_images]:
            img = Image.open(p).convert("RGB")
            x = TRANSFORM(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                y = int(model(x).argmax(-1).item())
            out.append((img, y))
        return out

    raise RuntimeError(
        "No ImageNet source available. Set IMAGENET_ROOT or run "
        "`huggingface-cli login` with ImageNet access."
    )


raw_samples = load_imagenet_subset(TIER_A)
print(f"Loaded {len(raw_samples)} raw images")

# ── Tensorize + pick target label ──
# Target = ground truth if model assigns it >5% probability; else model's top-1.
dataset = []
with torch.no_grad():
    for i, (pil, gt) in enumerate(tqdm(raw_samples, desc="prep")):
        x = TRANSFORM(pil).unsqueeze(0).to(DEVICE)
        probs = model(x).softmax(-1)[0]
        top1 = int(probs.argmax())
        target = gt if probs[gt].item() > 0.05 else top1
        dataset.append({
            "idx":       i,
            "x":         x,
            "target":    target,
            "gt_label":  gt,
            "p_target":  float(probs[target]),
            "label_str": imagenet_labels[target],
        })

# Nested tier subsets
tier_A_idx = list(range(min(TIER_A, len(dataset))))
tier_B_idx = tier_A_idx[:TIER_B]
tier_C_idx = tier_A_idx[:TIER_C]
print(f"Tier sizes: A={len(tier_A_idx)}  B={len(tier_B_idx)}  C={len(tier_C_idx)}")


## 5. Compute attributions for Tier A (checkpointed)

KL-IG is expensive — we cache every image's maps to `eval_cache/attributions.pkl`. Re-running this cell only recomputes missing keys. Delete the cache file to force a fresh run.


In [ ]:
def _load_attr_cache():
    if ATTR_CACHE.exists():
        with open(ATTR_CACHE, "rb") as f:
            return pickle.load(f)
    return {}

def _save_attr_cache(cache):
    with open(ATTR_CACHE, "wb") as f:
        pickle.dump(cache, f)

attr_cache = _load_attr_cache()
print(f"Loaded {len(attr_cache)} cached attributions from {ATTR_CACHE}")

results = {}  # idx -> {maps: {method: (H,W) cpu tensor}, klig_completeness, x, target, ...}

for row in tqdm(dataset[:TIER_A], desc="attr"):
    idx = row["idx"]
    key = f"img{idx:04d}"
    if key in attr_cache:
        cached = attr_cache[key]
        results[idx] = {**row,
                        "maps": cached["maps"],
                        "klig_completeness": cached["klig_completeness"]}
        continue

    maps, klig_res = compute_all(model, row["x"], row["target"])
    maps_cpu = {m: a.detach().cpu() for m, a in maps.items()}
    entry = {"maps": maps_cpu,
             "klig_completeness": float(klig_res._r.completeness_check())}
    attr_cache[key] = entry
    results[idx] = {**row, **entry}

    # Checkpoint every 10 images
    if (len(attr_cache) % 10) == 0:
        _save_attr_cache(attr_cache)

_save_attr_cache(attr_cache)

print(f"Attributions ready for {len(results)} images")
gaps = [r["klig_completeness"] for r in results.values()]
print(f"KL-IG sum(attr) summary: mean={np.mean(gaps):+.4f}  std={np.std(gaps):.4f}")


### Preview: first few images × all methods


In [ ]:
N_PREVIEW = min(4, len(results))
n_cols = 1 + len(methods)
fig, axes = plt.subplots(N_PREVIEW, n_cols,
                         figsize=(2.8 * n_cols, 3.0 * N_PREVIEW),
                         squeeze=False)
for row_i in range(N_PREVIEW):
    r = results[row_i]
    img_np = denormalize(r["x"][0]).detach().cpu().permute(1, 2, 0).numpy()
    axes[row_i][0].imshow(img_np)
    axes[row_i][0].set_title(f"{r['label_str'][:20]}\nP={r['p_target']:.2f}", fontsize=8)
    axes[row_i][0].axis("off")
    for col, method in enumerate(methods):
        amap = r["maps"][method]
        axes[row_i][col + 1].imshow(_attr_to_rgb(amap, clip_percentile=CLIP_PCT))
        axes[row_i][col + 1].set_title(method if row_i == 0 else "", fontsize=9)
        axes[row_i][col + 1].axis("off")
fig.suptitle("Attribution preview", fontweight="bold", fontsize=13)
plt.tight_layout(); plt.show()


---
## Tier A — cheap metrics (n = TIER_A)

Per-image metrics that only require forward passes through the network (no attribution reruns). Reported as mean ± std + boxplots across the tier.

- **Sensitivity-n** (Ancona et al. 2018, zero baseline)
- **IROF Faithfulness** (BEExAI)
- **Occlusion correlation** (Spearman ρ)
- **Insertion / Deletion** (blur baseline)
- **Insertion / Deletion** (N(0, 1) noise baseline)
- **Complexity** (Shannon entropy)
- **Signed insertion** + **sign agreement with IG-zero**


In [ ]:
# ── Sensitivity-n (Ancona et al. 2018) ──
# PCC between sum(attr[S]) and f(x) - f(x[x_S = 0])  with ZERO baseline.

def sensitivity_n(model, x, attr_map, target,
                  fractions=SENS_FRACTIONS, n_subsets=N_SENS_SUBSETS):
    H, W = attr_map.shape
    n_pix = H * W
    attr_flat = attr_map.detach().cpu().numpy().ravel()
    with torch.no_grad():
        f_orig = model(x).softmax(-1)[0, target].item()
    pccs = []
    for frac in fractions:
        n = max(1, int(frac * n_pix))
        attr_sums, output_diffs = [], []
        for _ in range(n_subsets):
            subset = np.random.choice(n_pix, size=n, replace=False)
            attr_sums.append(attr_flat[subset].sum())
            x_masked = x.clone()
            hi, wi = subset // W, subset % W
            x_masked[0, :, hi, wi] = 0.0        # faithful zero baseline
            with torch.no_grad():
                f_masked = model(x_masked).softmax(-1)[0, target].item()
            output_diffs.append(f_orig - f_masked)
        r, _ = stats.pearsonr(attr_sums, output_diffs)
        pccs.append(r if not np.isnan(r) else 0.0)
    return float(np.mean(pccs))


sens_n = {m: [] for m in methods}
for idx in tqdm(tier_A_idx, desc="sens-n"):
    r = results[idx]
    for m in methods:
        amap = r["maps"][m].to(DEVICE)
        sens_n[m].append(sensitivity_n(model, r["x"], amap, r["target"]))

tier_report("Sensitivity-n (PCC)", sens_n, higher_better=True)
tier_boxplot(sens_n, "PCC",
             f"Sensitivity-n (higher = more faithful, n={len(tier_A_idx)})")


In [ ]:
# ── IROF Faithfulness (BEExAI) ──
# Remove patches most to least important, measure AUC of P(target) curve.

def irof_faithfulness(model, x, attr_map, target,
                      patch_size=IROF_PATCH, n_steps=IROF_STEPS):
    H, W = attr_map.shape
    a = attr_map.detach().cpu().abs().numpy()
    patches = []
    for h in range(0, H - patch_size + 1, patch_size):
        for w in range(0, W - patch_size + 1, patch_size):
            patches.append((a[h:h+patch_size, w:w+patch_size].mean(), h, w))
    patches.sort(reverse=True)  # most important first
    baseline = make_blur_baseline(x)
    cur = x.clone()
    scores = []
    step_size = max(1, len(patches) // n_steps)
    with torch.no_grad():
        scores.append(model(cur).softmax(-1)[0, target].item())
        for i in range(0, len(patches), step_size):
            for _, h, w in patches[i:i+step_size]:
                cur[0, :, h:h+patch_size, w:w+patch_size] = \
                    baseline[0, :, h:h+patch_size, w:w+patch_size]
            scores.append(model(cur).softmax(-1)[0, target].item())
    return float(np.trapezoid(np.array(scores), dx=1.0 / len(scores)))


irof = {m: [] for m in methods}
for idx in tqdm(tier_A_idx, desc="IROF"):
    r = results[idx]
    for m in methods:
        irof[m].append(irof_faithfulness(model, r["x"], r["maps"][m], r["target"]))

tier_report("IROF AUC", irof, higher_better=False)
tier_boxplot(irof, "IROF AUC",
             f"Faithfulness (lower = faster drop = better, n={len(tier_A_idx)})")


In [ ]:
# ── Occlusion correlation ──
# Spearman between an occlusion-based saliency map and the method's map.

def occlusion_map(model, x, target, ps=OCCLUSION_PATCH, stride=OCCLUSION_STRIDE):
    _, C, H, W = x.shape
    with torch.no_grad():
        f0 = model(x)[0, target].item()
    hs = list(range(0, H - ps + 1, stride))
    ws = list(range(0, W - ps + 1, stride))
    om = np.zeros((len(hs), len(ws)))
    with torch.no_grad():
        for i, h in enumerate(hs):
            for j, w in enumerate(ws):
                mk = x.clone(); mk[0, :, h:h+ps, w:w+ps] = 0
                om[i, j] = f0 - model(mk)[0, target].item()
    return om


def attr_to_grid(attr_map, ps, stride, H, W):
    a = attr_map.detach().cpu().abs().numpy()
    hs = list(range(0, H - ps + 1, stride))
    ws = list(range(0, W - ps + 1, stride))
    g = np.zeros((len(hs), len(ws)))
    for i, h in enumerate(hs):
        for j, w in enumerate(ws):
            g[i, j] = a[h:h+ps, w:w+ps].mean()
    return g


occ = {m: [] for m in methods}
for idx in tqdm(tier_A_idx, desc="occlusion"):
    r = results[idx]
    om = occlusion_map(model, r["x"], r["target"])
    _, _, H, W = r["x"].shape
    for m in methods:
        ag = attr_to_grid(r["maps"][m], OCCLUSION_PATCH, OCCLUSION_STRIDE, H, W)
        rho, _ = stats.spearmanr(om.ravel(), ag.ravel())
        occ[m].append(0.0 if np.isnan(rho) else rho)

tier_report("Occlusion Spearman ρ", occ, higher_better=True)
tier_boxplot(occ, "Spearman ρ",
             f"Occlusion Correlation (higher = better, n={len(tier_A_idx)})")


In [ ]:
# ── Insertion / Deletion (blur baseline + noise baseline) ──

def insertion_deletion(model, x, attr_map, target, substrate, n_steps=N_INSERTION_STEPS):
    H, W = attr_map.shape
    n_pix = H * W
    order = attr_map.detach().cpu().abs().view(-1).argsort(descending=True)
    pps = max(1, n_pix // n_steps)
    ins_img, del_img = substrate.clone(), x.clone()
    ins_scores, del_scores = [], []
    with torch.no_grad():
        ins_scores.append(model(ins_img).softmax(-1)[0, target].item())
        del_scores.append(model(del_img).softmax(-1)[0, target].item())
        for step in range(1, n_steps + 1):
            s, e = (step-1)*pps, min(step*pps, n_pix)
            if s >= n_pix:
                ins_scores.append(ins_scores[-1]); del_scores.append(del_scores[-1]); continue
            idx = order[s:e]
            hi, wi = idx // W, idx % W
            ins_img[0, :, hi, wi] = x[0, :, hi, wi]
            del_img[0, :, hi, wi] = substrate[0, :, hi, wi]
            ins_scores.append(model(ins_img).softmax(-1)[0, target].item())
            del_scores.append(model(del_img).softmax(-1)[0, target].item())
    ins = np.array(ins_scores); dl = np.array(del_scores)
    return float(np.trapezoid(ins, dx=1/n_steps)), float(np.trapezoid(dl, dx=1/n_steps))


ins_blur = {m: [] for m in methods}
del_blur = {m: [] for m in methods}
ins_noise = {m: [] for m in methods}
del_noise = {m: [] for m in methods}

for idx in tqdm(tier_A_idx, desc="ins/del"):
    r = results[idx]
    blur = make_blur_baseline(r["x"])
    noise = torch.randn_like(r["x"])
    for m in methods:
        amap = r["maps"][m]
        ib, db = insertion_deletion(model, r["x"], amap, r["target"], blur)
        ins_blur[m].append(ib); del_blur[m].append(db)
        inz, dnz = insertion_deletion(model, r["x"], amap, r["target"], noise)
        ins_noise[m].append(inz); del_noise[m].append(dnz)

tier_report("Insertion AUC (blur)",  ins_blur, higher_better=True)
tier_report("Deletion AUC (blur)",   del_blur, higher_better=False)
tier_report("Insertion AUC (noise)", ins_noise, higher_better=True)
tier_report("Deletion AUC (noise)",  del_noise, higher_better=False)

# Four-panel boxplot
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
panels = [
    (axes[0, 0], ins_blur,  f"Insertion (blur, higher = better, n={len(tier_A_idx)})"),
    (axes[0, 1], del_blur,  f"Deletion (blur, lower = better, n={len(tier_A_idx)})"),
    (axes[1, 0], ins_noise, f"Insertion (noise, higher = better, n={len(tier_A_idx)})"),
    (axes[1, 1], del_noise, f"Deletion (noise, lower = better, n={len(tier_A_idx)})"),
]
for ax, data, title in panels:
    bp = ax.boxplot([data[m] for m in methods], labels=methods, patch_artist=True,
                    medianprops=dict(color="black", lw=1.5))
    for p, m in zip(bp["boxes"], methods):
        p.set_facecolor(COLORS[m]); p.set_alpha(0.8)
    ax.set_title(title, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()


In [ ]:
# ── Complexity (Shannon entropy of |attribution| distribution) ──

def beexai_complexity(attr_map):
    a = np.abs(attr_map.detach().cpu().numpy().ravel())
    total = a.sum()
    if total < 1e-12:
        return 0.0
    p = a / total
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))


entropy = {m: [] for m in methods}
for idx in tqdm(tier_A_idx, desc="complexity"):
    r = results[idx]
    for m in methods:
        entropy[m].append(beexai_complexity(r["maps"][m]))

tier_report("Shannon entropy (bits)", entropy, higher_better=False)
tier_boxplot(entropy, "Entropy (bits)",
             f"Complexity (lower = more concentrated, n={len(tier_A_idx)})")


In [ ]:
# ── Signed Attribution Evaluation ──
# (1) Signed insertion: AUC when inserting only positive- vs only negative-attributed pixels.
#     Large gap (pos − neg) → signs carry class-discriminating information.
# (2) Sign agreement with IG-zero (reference).

def signed_insertion(model, x, attr_map, target, n_steps=50, positive=True):
    H, W = attr_map.shape
    a = attr_map.detach().cpu().numpy().ravel()
    if positive:
        valid = np.where(a > 0)[0]
        order = valid[np.argsort(-a[valid])]
    else:
        valid = np.where(a < 0)[0]
        order = valid[np.argsort(a[valid])]
    if len(order) == 0:
        return 0.0
    substrate = make_blur_baseline(x)
    img = substrate.clone()
    scores = []
    pps = max(1, len(order) // n_steps)
    with torch.no_grad():
        scores.append(model(img).softmax(-1)[0, target].item())
        for s in range(1, n_steps + 1):
            chunk = order[(s-1)*pps: s*pps]
            if len(chunk) == 0:
                scores.append(scores[-1]); continue
            hi, wi = chunk // W, chunk % W
            img[0, :, hi, wi] = x[0, :, hi, wi]
            scores.append(model(img).softmax(-1)[0, target].item())
    return float(np.trapezoid(np.array(scores), dx=1.0/len(scores)))


def sign_agreement(attr_a, attr_b, eps=1e-8):
    a = attr_a.detach().cpu().numpy().ravel()
    b = attr_b.detach().cpu().numpy().ravel()
    valid = (np.abs(a) > eps) & (np.abs(b) > eps)
    if valid.sum() == 0:
        return 0.0
    return float((np.sign(a[valid]) == np.sign(b[valid])).mean())


sign_disc = {m: [] for m in methods}      # AUC(+) − AUC(−)
sign_agree_ig = {m: [] for m in methods}  # vs IG-zero map

for idx in tqdm(tier_A_idx, desc="signed"):
    r = results[idx]
    ig_map = r["maps"]["IG-zero"]
    for m in methods:
        amap = r["maps"][m]
        pos = signed_insertion(model, r["x"], amap, r["target"], positive=True)
        neg = signed_insertion(model, r["x"], amap, r["target"], positive=False)
        sign_disc[m].append(pos - neg)
        sign_agree_ig[m].append(sign_agreement(amap, ig_map))

tier_report("Sign discrimination gap (+AUC − −AUC)", sign_disc, higher_better=True)
tier_report("Sign agreement with IG-zero", sign_agree_ig, higher_better=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title in [
    (axes[0], sign_disc,    f"Sign discrimination gap (higher = better, n={len(tier_A_idx)})"),
    (axes[1], sign_agree_ig, f"Sign agreement with IG-zero (n={len(tier_A_idx)})"),
]:
    bp = ax.boxplot([data[m] for m in methods], labels=methods, patch_artist=True,
                    medianprops=dict(color="black", lw=1.5))
    for p, m in zip(bp["boxes"], methods):
        p.set_facecolor(COLORS[m]); p.set_alpha(0.8)
    ax.set_title(title, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()


---
## Tier B — expensive metrics (n = TIER_B)

Metrics that require **attribution reruns per image** (Max-Sensitivity, Perturbation Robustness) or per-object computation (Semantic Focus).

- **Max-Sensitivity Robustness** — reruns each method k times under small input noise
- **Perturbation Robustness** — Spearman ρ vs clean map across noise σ
- **Semantic Focus Score** — attribution mass on object vs background (GrabCut)


In [ ]:
# ── Max-Sensitivity Robustness ──
# Each method rerun ROB_TRIALS times per image under small input noise.
# Reported: max relative change in attribution.

def max_sensitivity(model, x, clean_map, target, method,
                    eps=ROB_EPS, n_trials=ROB_TRIALS):
    a_orig = clean_map.detach().cpu().float()
    norm_orig = a_orig.norm().item()
    if norm_orig < 1e-10:
        return 0.0
    max_rel = 0.0
    for _ in range(n_trials):
        x_pert = x + eps * torch.randn_like(x)
        a_pert = COMPUTE_FN[method](model, x_pert, target).detach().cpu().float()
        rel = (a_orig - a_pert).norm().item() / norm_orig
        max_rel = max(max_rel, rel)
    return max_rel


rob = {m: [] for m in methods}
for idx in tqdm(tier_B_idx, desc="max-sens"):
    r = results[idx]
    for m in methods:
        rob[m].append(max_sensitivity(model, r["x"], r["maps"][m], r["target"], m))

tier_report("Max-Sensitivity", rob, higher_better=False)
tier_boxplot(rob, "Max-Sensitivity",
             f"Robustness (lower = more stable, n={len(tier_B_idx)})")


In [ ]:
# ── Perturbation Robustness ──
# Spearman ρ vs clean map across PERTURBATION_SIGMAS, PERTURBATION_RUNS reruns each.

def perturbation_robustness_single(model, x, target, clean_map, method,
                                   sigmas=PERTURBATION_SIGMAS, n_runs=PERTURBATION_RUNS):
    out = {}
    for sigma in sigmas:
        corrs = []
        for _ in range(n_runs):
            x_noisy = x + sigma * torch.randn_like(x)
            nm = COMPUTE_FN[method](model, x_noisy, target).detach().cpu().float()
            corr, _ = stats.spearmanr(clean_map.detach().cpu().numpy().ravel(),
                                      nm.numpy().ravel())
            corrs.append(0.0 if np.isnan(corr) else corr)
        out[sigma] = float(np.mean(corrs))
    return out


perturb = {m: {s: [] for s in PERTURBATION_SIGMAS} for m in methods}
for idx in tqdm(tier_B_idx, desc="perturb"):
    r = results[idx]
    for m in methods:
        res = perturbation_robustness_single(model, r["x"], r["target"],
                                             r["maps"][m], m)
        for s, v in res.items():
            perturb[m][s].append(v)

fig, ax = plt.subplots(figsize=(10, 5))
for m in methods:
    means = [np.mean(perturb[m][s]) for s in PERTURBATION_SIGMAS]
    stds  = [np.std(perturb[m][s])  for s in PERTURBATION_SIGMAS]
    ax.errorbar(PERTURBATION_SIGMAS, means, yerr=stds, fmt="o-",
                label=m, color=COLORS[m], lw=2, capsize=3)
ax.set(xlabel="Perturbation σ", ylabel="Spearman ρ with clean",
       title=f"Perturbation Robustness (higher = more stable, n={len(tier_B_idx)})",
       ylim=(0, 1.05))
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\nMean Spearman ρ at σ = 0.05:")
for m in methods:
    v = np.array(perturb[m][0.05])
    print(f"  {m:14s}: {v.mean():+.3f} ± {v.std():.3f}")


In [ ]:
# ── Semantic Focus Score (GrabCut-based object mask) ──

def estimate_object_mask(x, attr_map, use_grabcut=True):
    import cv2
    H, W = attr_map.shape
    a = np.abs(attr_map.detach().cpu().numpy())
    seed = (a >= np.percentile(a, 80)).astype(np.uint8)
    if not use_grabcut:
        return seed
    img_rgb = denormalize(x[0]).detach().cpu().permute(1, 2, 0).numpy()
    img_bgr = (img_rgb * 255).clip(0, 255).astype(np.uint8)[:, :, ::-1].copy()
    gc_mask = np.where(seed, cv2.GC_PR_FGD, cv2.GC_PR_BGD).astype(np.uint8)
    top5 = (a >= np.percentile(a, 95)).astype(np.uint8)
    gc_mask[top5 == 1] = cv2.GC_FGD
    edge_mask = np.zeros((H, W), dtype=np.uint8)
    border = max(H, W) // 10
    edge_mask[:border, :] = 1; edge_mask[-border:, :] = 1
    edge_mask[:, :border] = 1; edge_mask[:, -border:] = 1
    low_attr = (a < np.percentile(a, 10)).astype(np.uint8)
    gc_mask[(edge_mask == 1) & (low_attr == 1)] = cv2.GC_BGD
    try:
        bgd_model = np.zeros((1, 65), np.float64)
        fgd_model = np.zeros((1, 65), np.float64)
        cv2.grabCut(img_bgr, gc_mask, None, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_MASK)
        final_mask = np.where((gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD), 1, 0).astype(np.uint8)
    except Exception:
        final_mask = seed
    return final_mask


def object_focus_ratio(attr_map, object_mask):
    a = np.abs(attr_map.detach().cpu().numpy())
    total = a.sum()
    if total < 1e-12:
        return 0.0
    return float(a[object_mask == 1].sum() / total)


def discriminative_concentration(attr_map, object_mask, top_pct=10):
    a = np.abs(attr_map.detach().cpu().numpy())
    obj_vals = a[object_mask == 1]
    if len(obj_vals) == 0 or obj_vals.sum() < 1e-12:
        return 0.0
    k = max(1, int(top_pct / 100 * len(obj_vals)))
    top_k_sum = np.sort(obj_vals)[::-1][:k].sum()
    return float(top_k_sum / obj_vals.sum())


ofr = {m: [] for m in methods}
dconc = {m: [] for m in methods}
for idx in tqdm(tier_B_idx, desc="semantic"):
    r = results[idx]
    # Use KL-IG map as seed; GrabCut refines against RGB image
    obj_mask = estimate_object_mask(r["x"], r["maps"]["KL-IG"], use_grabcut=True)
    for m in methods:
        amap = r["maps"][m]
        ofr[m].append(object_focus_ratio(amap, obj_mask))
        dconc[m].append(discriminative_concentration(amap, obj_mask, top_pct=10))

tier_report("Object Focus Ratio (OFR)", ofr, higher_better=True)
tier_report("Discriminative Concentration (top-10%)", dconc, higher_better=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title in [
    (axes[0], ofr,   f"Object Focus Ratio (higher = better, n={len(tier_B_idx)})"),
    (axes[1], dconc, f"Discriminative Concentration (higher = more focused, n={len(tier_B_idx)})"),
]:
    bp = ax.boxplot([data[m] for m in methods], labels=methods, patch_artist=True,
                    medianprops=dict(color="black", lw=1.5))
    for p, m in zip(bp["boxes"], methods):
        p.set_facecolor(COLORS[m]); p.set_alpha(0.8)
    ax.set_title(title, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()


---
## Tier C — KL-IG variance / convergence (n = TIER_C)

KL-IG-specific tests that require multiple reruns or hyperparameter sweeps per image.

- **Sign stability of KL-IG** — fraction of active pixels with reproducible sign across MC reruns
- **Completeness vs MC samples** — does `|sum(attr) − ΔE[f]|` shrink ~ 1/√N?


In [ ]:
# ── KL-IG sign stability across MC reruns ──
# For each image, run KL-IG n_runs times (each with its own MC noise),
# and measure per-pixel fraction of runs agreeing with the majority sign.

def sign_stability_klig(model, x, target, n_runs=5, eps=1e-8):
    maps = []
    for _ in range(n_runs):
        m = compute_klig(model, x, target).attr_map("absmax").detach().cpu().numpy()
        maps.append(m)
    stack = np.stack(maps)              # (n_runs, H, W)
    signs = np.sign(stack)
    majority = np.sign(signs.sum(axis=0))
    agree = (signs == majority[None, :, :]).mean(axis=0)
    active = np.abs(stack).mean(axis=0) > eps
    if active.sum() == 0:
        return 0.0
    return float(agree[active].mean())


sign_stab_vals = []
for idx in tqdm(tier_C_idx, desc="sign-stability"):
    r = results[idx]
    sign_stab_vals.append(sign_stability_klig(model, r["x"], r["target"], n_runs=5))

sign_stab_vals = np.array(sign_stab_vals)
print(f"\nKL-IG sign stability across 5 MC reruns  (n={len(tier_C_idx)} images)")
print(f"  mean   = {sign_stab_vals.mean():.3f} ± {sign_stab_vals.std():.3f}")
print(f"  median = {np.median(sign_stab_vals):.3f}")
print(f"  min    = {sign_stab_vals.min():.3f}   max = {sign_stab_vals.max():.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(sign_stab_vals, bins=10, color=COLORS["KL-IG"], edgecolor="black")
ax.set(xlabel="Fraction of active pixels with reproducible sign",
       ylabel="# images",
       title=f"KL-IG Sign Stability (n={len(tier_C_idx)}, 5 reruns each)")
ax.axvline(0.5, color="gray", ls="--", alpha=0.5, label="chance")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Completeness vs MC Samples ──
# Gap  |sum(attr) − ΔE[f]|  should shrink ~ 1/√N as MC samples per step grow.

def klig_with_n_samples(model, x, target, n_samples,
                        n_steps=N_STEPS, sigma_final=None):
    if sigma_final is None:
        sigma_final = max(find_sigma_stop(model, x, target=target, tau=0.95), 1.0/256.0)
    attr = ImageAttributor(model=model, n_steps=n_steps, n_samples=n_samples,
                           sigma_final=sigma_final, device=DEVICE)
    return attr.attribute(x, target=target, show_progress=False)


def output_diff_reference(model, x, target, sigma_final, n_mc=200):
    """E[f(x + sigma*eps)] - E[f(z)]  where z ~ N(0, I)."""
    with torch.no_grad():
        eps = torch.randn(n_mc, *x.squeeze(0).shape, device=x.device)
        x_final = x + sigma_final * eps
        f_final = model(x_final).softmax(-1)[:, target].mean().item()
        z = torch.randn(n_mc, *x.squeeze(0).shape, device=x.device)
        f_start = model(z).softmax(-1)[:, target].mean().item()
    return f_final - f_start


N_SAMPLES_SWEEP = [1, 2, 5, 10, 20, 40]
N_TRIALS = 3

# (n_imgs, n_settings, n_trials)
gap_matrix = np.zeros((len(tier_C_idx), len(N_SAMPLES_SWEEP), N_TRIALS))
sum_matrix = np.zeros_like(gap_matrix)
ref_vec = np.zeros(len(tier_C_idx))

for i, idx in enumerate(tqdm(tier_C_idx, desc="comp vs MC")):
    r = results[idx]
    x, t = r["x"], r["target"]
    sigma_final = max(find_sigma_stop(model, x, target=t, tau=0.95), 1.0/256.0)
    ref = output_diff_reference(model, x, t, sigma_final, n_mc=300)
    ref_vec[i] = ref
    for j, ns in enumerate(N_SAMPLES_SWEEP):
        for k in range(N_TRIALS):
            kr = klig_with_n_samples(model, x, t, n_samples=ns,
                                     n_steps=N_STEPS, sigma_final=sigma_final)
            s = float(kr._r.completeness_check())
            sum_matrix[i, j, k] = s
            gap_matrix[i, j, k] = abs(s - ref)

mean_gap = gap_matrix.mean(axis=(0, 2))
std_gap  = gap_matrix.std(axis=(0, 2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) sum(attr) across trials with reference band
for i in range(len(tier_C_idx)):
    mu = sum_matrix[i].mean(axis=1)
    axes[0].plot(N_SAMPLES_SWEEP, mu, "o-", alpha=0.3, color=COLORS["KL-IG"])
    axes[0].axhline(ref_vec[i], color="gray", ls=":", alpha=0.3)
axes[0].set(xlabel="N_SAMPLES (MC)", ylabel="Σ attributions (per image)",
            title=f"Completeness convergence (n={len(tier_C_idx)} imgs)")
axes[0].set_xscale("log"); axes[0].grid(alpha=0.3, which="both")

# (b) absolute gap vs reference, averaged across images
axes[1].errorbar(N_SAMPLES_SWEEP, mean_gap, yerr=std_gap, fmt="o-",
                 color=COLORS["KL-IG"], lw=2, capsize=4, label="KL-IG gap")
if mean_gap[0] > 0:
    ref_line = mean_gap[0] * np.sqrt(N_SAMPLES_SWEEP[0] / np.array(N_SAMPLES_SWEEP))
    axes[1].plot(N_SAMPLES_SWEEP, ref_line, "k--", alpha=0.5, label="$1/\\sqrt{N}$ reference")
axes[1].set(xlabel="N_SAMPLES (MC)", ylabel="| Σ attr − ΔE[f] |",
            title=f"Completeness gap vs MC samples ({N_TRIALS} trials / img)")
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].legend(); axes[1].grid(alpha=0.3, which="both")

plt.suptitle("KL-IG Completeness vs Monte Carlo Samples", fontweight="bold", fontsize=12)
plt.tight_layout(); plt.show()

print("\nN_SAMPLES -> mean |gap| (averaged over images + trials)")
for ns, g, s in zip(N_SAMPLES_SWEEP, mean_gap, std_gap):
    print(f"  N={ns:3d}: {g:.5f} ± {s:.5f}")
print(f"\nImprovement factor: {mean_gap[0] / (mean_gap[-1] + 1e-12):.2f}×")


---
## Summary: headline table + JSON dump


In [ ]:
def fmt(vals):
    arr = np.array(vals)
    return f"{arr.mean():+.3f} ± {arr.std():.3f}"


summary = {
    # Tier A
    "Sensitivity-n (↑)":   {m: fmt(sens_n[m])          for m in methods},
    "IROF AUC (↓)":        {m: fmt(irof[m])            for m in methods},
    "Occlusion ρ (↑)":     {m: fmt(occ[m])             for m in methods},
    "Insertion blur (↑)":  {m: fmt(ins_blur[m])        for m in methods},
    "Deletion blur (↓)":   {m: fmt(del_blur[m])        for m in methods},
    "Insertion noise (↑)": {m: fmt(ins_noise[m])       for m in methods},
    "Deletion noise (↓)":  {m: fmt(del_noise[m])       for m in methods},
    "Entropy (↓)":         {m: fmt(entropy[m])         for m in methods},
    "Sign gap (↑)":        {m: fmt(sign_disc[m])       for m in methods},
    "Sign ≡ IG (↑)":       {m: fmt(sign_agree_ig[m])   for m in methods},
    # Tier B
    "Max-Sens (↓)":        {m: fmt(rob[m])             for m in methods},
    "Object Focus (↑)":    {m: fmt(ofr[m])             for m in methods},
    "Discrim Conc (↑)":    {m: fmt(dconc[m])           for m in methods},
}

# ── Print aligned ASCII table ──
col_w = 18
header = f"{'Metric':22s}" + "".join(f"{m:>{col_w}s}" for m in methods)
print(header)
print("-" * len(header))
for metric, row in summary.items():
    print(f"{metric:22s}" + "".join(f"{row[m]:>{col_w}s}" for m in methods))

# ── Tier C (single-column since KL-IG-specific) ──
print("\nTier C (KL-IG specific):")
print(f"  Sign stability (5 reruns): {sign_stab_vals.mean():.3f} ± {sign_stab_vals.std():.3f}")
print(f"  Completeness gap @ N=1:    {mean_gap[0]:.5f}")
print(f"  Completeness gap @ N={N_SAMPLES_SWEEP[-1]}:   {mean_gap[-1]:.5f}")
print(f"  Gap improvement factor:    {mean_gap[0] / (mean_gap[-1] + 1e-12):.2f}×")

# ── Persist ──
full = {
    "tier_sizes": {"A": len(tier_A_idx), "B": len(tier_B_idx), "C": len(tier_C_idx)},
    "summary": summary,
    "tier_C": {
        "sign_stability_mean": float(sign_stab_vals.mean()),
        "sign_stability_std":  float(sign_stab_vals.std()),
        "mc_gap_mean":         mean_gap.tolist(),
        "mc_gap_std":          std_gap.tolist(),
        "n_samples_sweep":     N_SAMPLES_SWEEP,
    },
}
with open(CHECKPOINT_DIR / "summary.json", "w") as f:
    json.dump(full, f, indent=2)
print(f"\nSaved → {CHECKPOINT_DIR / 'summary.json'}")
